In [20]:
import os
import time
import glob
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator
from pathlib import Path

Configuration

In [21]:
RUNS_DIR = "runs/detect"          # dictionary with trained modles
TEST_IMAGE = "C:\\Users\\Dawid\\Desktop\\praca inżynierska\\real-time-alaysis-research\\scripts\\YOLO\\test.png"     # single image for inference test
TEST_DATASET = os.path.abspath("../../datasets/Person&Head_DataSet/dataset.yaml")        # dataset YAML do model.val()
FPS_WARMUP = 5                    # number of warm-up inference runs
FPS_RUNS = 30                     # number of FPS measurement runs
IMG_SIZE = 640                    # image size for testing
CLASSES = ["human", "human_head"] # names of your classes

MODELS = {
    "YOLOv8n":  "yolov8n_V1_S10000_E100_IMG640_B16",
    "YOLOv8m":  "yolov8m_V1_S10000_E150_IMG800_B16",
    "YOLOv10s": "yolov10s_V1_S10000_E100_IMG640_B32",
    "YOLOv10m": "yolov10m_V1_S10000_E150_IMG800_B16",
    "YOLOv11s": "yolo11s_V1_S10000_E150_IMG640_B32",
    "YOLOv11m": "yolo11m_V1_S10000_E150_IMG800_B16",
    "YOLO26s":  "yolo26s_V1_S10000_E150_IMG640_B32",
    "YOLO26m":  "yolo26m_V1_S10000_E150_IMG800_B16",
}
 
# colors for plotting, one per model
GEN_COLORS = {
    "YOLOv8n":  "#378ADD", "YOLOv8m":  "#185FA5",
    "YOLOv10s": "#EF9F27", "YOLOv10m": "#BA7517",
    "YOLOv11s": "#1D9E75", "YOLOv11m": "#0F6E56",
    "YOLO26s":  "#7F77DD", "YOLO26m":  "#534AB7",
}
 
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
 
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "figure.dpi": 150,
})

Functions

In [22]:
def get_best_weights(model_folder: str) -> str | None:
    """Znajdź best.pt w folderze modelu."""
    path = Path(RUNS_DIR) / model_folder / "weights" / "best.pt"
    if path.exists():
        return str(path)
    candidates = glob.glob(f"{RUNS_DIR}/{model_folder}*/weights/best.pt")
    return candidates[0] if candidates else None


def get_results_csv(model_folder: str) -> pd.DataFrame | None:
    """Wczytaj results.csv z treningu."""
    path = Path(RUNS_DIR) / model_folder / "results.csv"
    if not path.exists():
        candidates = glob.glob(f"{RUNS_DIR}/{model_folder}*/results.csv")
        if not candidates:
            return None
        path = candidates[0]
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df


def set_model_inference_mode(model):
    """Ustaw model w trybie inferencyjnym, minimalizując konflikt wersjonowania tensorów."""
    try:
        if hasattr(model, "eval"):
            model.eval()
    except Exception:
        pass
    try:
        if hasattr(model, "model") and hasattr(model.model, "eval"):
            model.model.eval()
    except Exception:
        pass


def measure_fps(weights_path: str, img_path: str, device: str, n_runs: int = FPS_RUNS) -> float:
    """Zmierz FPS na danym urządzeniu bez trybu grad."""
    from ultralytics import YOLO
    import torch

    model = YOLO(weights_path)
    model.to(device)
    set_model_inference_mode(model)

    with torch.inference_mode():
        for _ in range(FPS_WARMUP):
            model.predict(img_path, imgsz=IMG_SIZE, verbose=False, device=device)

        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            model.predict(img_path, imgsz=IMG_SIZE, verbose=False, device=device)
            times.append(time.perf_counter() - t0)

    if device.startswith("cuda"):
        torch.cuda.empty_cache()

    return round(1.0 / np.mean(times), 1)


def get_model_size_mb(weights_path: str) -> float:
    return round(Path(weights_path).stat().st_size / 1e6, 1)


def bold_max(series: pd.Series) -> list[str]:
    """Podkreśl najlepszą wartość w kolumnie."""
    best = series.max()
    return ["**" + str(v) + "**" if v == best else str(v) for v in series]

In [23]:
def collect_metrics() -> pd.DataFrame:
    try:
        from ultralytics import YOLO
        import torch
        has_gpu = torch.cuda.is_available()
    except ImportError:
        print("BŁĄD: Zainstaluj ultralytics: pip install ultralytics")
        return pd.DataFrame()

    records = []

    for display_name, folder in MODELS.items():
        print(f"\n{'='*50}")
        print(f"  Przetwarzam: {display_name}")
        print(f"{'='*50}")

        weights = get_best_weights(folder)
        if not weights:
            print(f"  POMINIĘTO — brak best.pt w {folder}")
            continue

        model = YOLO(weights)
        set_model_inference_mode(model)
        size_mb = get_model_size_mb(weights)
        n_params = sum(p.numel() for p in model.model.parameters()) / 1e6

        # Walidacja — mAP, Precision, Recall
        print("  Ewaluacja na zbiorze testowym...")
        with torch.inference_mode():
            metrics = model.val(data=TEST_DATASET, imgsz=IMG_SIZE, verbose=False)

        map50     = round(float(metrics.box.map50), 4)
        map50_95  = round(float(metrics.box.map), 4)
        precision = round(float(metrics.box.mp), 4)
        recall    = round(float(metrics.box.mr), 4)
        f1        = round(2 * precision * recall / (precision + recall + 1e-8), 4)

        # mAP per klasa
        maps_per_class = metrics.box.maps  # np.array [human, human_head]
        map_human      = round(float(maps_per_class[0]), 4) if len(maps_per_class) > 0 else None
        map_head       = round(float(maps_per_class[1]), 4) if len(maps_per_class) > 1 else None

        # FPS CPU
        print("  Pomiar FPS na CPU...")
        fps_cpu = measure_fps(weights, TEST_IMAGE, "cpu")

        # FPS GPU
        fps_gpu = None
        if has_gpu:
            print("  Pomiar FPS na GPU...")
            fps_gpu = measure_fps(weights, TEST_IMAGE, "cuda:0")
        else:
            print("  GPU niedostępne — pomijam.")

        inf_ms_cpu = round(1000 / fps_cpu, 1)

        record = {
            "Model":            display_name,
            "mAP@0.5":          map50,
            "mAP@0.5:0.95":     map50_95,
            "Precision":        precision,
            "Recall":           recall,
            "F1":               f1,
            "AP_human":         map_human,
            "AP_human_head":    map_head,
            "FPS_CPU":          fps_cpu,
            "FPS_GPU":          fps_gpu,
            "Czas_ms_CPU":      inf_ms_cpu,
            "Rozmiar_MB":       size_mb,
            "Params_M":         round(n_params, 1),
        }
        records.append(record)
        print(f"  mAP@0.5={map50:.3f} | P={precision:.3f} | R={recall:.3f} | FPS_CPU={fps_cpu}")

    return pd.DataFrame(records)

In [24]:
def plot_map_vs_fps(df: pd.DataFrame):
    """Scatter: mAP@0.5 vs FPS CPU — kompromis dokładność/szybkość."""
    fig, ax = plt.subplots(figsize=(9, 6))
 
    for _, row in df.iterrows():
        color = GEN_COLORS.get(row["Model"], "#888")
        ax.scatter(row["FPS_CPU"], row["mAP@0.5"],
                   color=color, s=180, zorder=5, edgecolors="white", linewidth=1.5)
        ax.annotate(row["Model"],
                    (row["FPS_CPU"], row["mAP@0.5"]),
                    textcoords="offset points", xytext=(8, 4),
                    fontsize=9, color=color, fontweight="bold")
 
    ax.set_xlabel("FPS na CPU (im więcej tym lepiej →)", fontsize=12)
    ax.set_ylabel("mAP@0.5 (im wyżej tym lepiej ↑)", fontsize=12)
    ax.set_title("Kompromis: dokładność vs szybkość na CPU", fontsize=13, fontweight="bold")
 
    # Legenda generacji
    patches = [
        mpatches.Patch(color="#378ADD", label="YOLOv8"),
        mpatches.Patch(color="#EF9F27", label="YOLOv10"),
        mpatches.Patch(color="#1D9E75", label="YOLOv11"),
        mpatches.Patch(color="#7F77DD", label="YOLO26"),
    ]
    ax.legend(handles=patches, loc="lower right", fontsize=10)
    ax.annotate("Idealny model", xy=(df["FPS_CPU"].max()*0.92, df["mAP@0.5"].max()*0.98),
                fontsize=9, color="gray", style="italic")
 
    plt.tight_layout()
    out = OUTPUT_DIR / "wykres_mAP_vs_FPS.png"
    plt.savefig(out, bbox_inches="tight")
    plt.close()
    print(f"  Zapisano: {out}")
 
 
def plot_map_per_class(df: pd.DataFrame):
    """Grouped bar: AP_human vs AP_human_head dla każdego modelu."""
    if df["AP_human"].isna().all():
        print("  Brak danych per-klasa — pomijam wykres.")
        return
 
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(df))
    w = 0.35
    colors_h  = [GEN_COLORS.get(m, "#888") for m in df["Model"]]
    colors_hh = [c + "88" for c in colors_h]  # półprzezroczyste dla head
 
    bars1 = ax.bar(x - w/2, df["AP_human"],      w, label="human",      color=colors_h,  alpha=0.9)
    bars2 = ax.bar(x + w/2, df["AP_human_head"], w, label="human_head", color=colors_hh, alpha=0.9, hatch="//")
 
    ax.set_xticks(x)
    ax.set_xticklabels(df["Model"], rotation=20, ha="right", fontsize=10)
    ax.set_ylabel("AP@0.5", fontsize=12)
    ax.set_title("Dokładność per klasa: human vs human_head", fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=11)
 
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
 
    plt.tight_layout()
    out = OUTPUT_DIR / "wykres_mAP_per_klasa.png"
    plt.savefig(out, bbox_inches="tight")
    plt.close()
    print(f"  Zapisano: {out}")
 
 
def plot_fps_cpu_gpu(df: pd.DataFrame):
    """Grouped bar: FPS CPU vs GPU."""
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(df))
    w = 0.35
    colors = [GEN_COLORS.get(m, "#888") for m in df["Model"]]
 
    ax.bar(x - w/2, df["FPS_CPU"], w, label="CPU", color=colors, alpha=0.85)
    if df["FPS_GPU"].notna().any():
        ax.bar(x + w/2, df["FPS_GPU"].fillna(0), w, label="GPU",
               color=colors, alpha=0.4, hatch="xx")
        ax.axhline(15, color="red", linestyle="--", linewidth=1, alpha=0.7, label="Próg real-time (15 FPS)")
 
    ax.set_xticks(x)
    ax.set_xticklabels(df["Model"], rotation=20, ha="right", fontsize=10)
    ax.set_ylabel("Klatki na sekundę [FPS]", fontsize=12)
    ax.set_title("Wydajność: CPU vs GPU", fontsize=13, fontweight="bold")
    ax.legend(fontsize=11)
 
    plt.tight_layout()
    out = OUTPUT_DIR / "wykres_FPS_CPU_GPU.png"
    plt.savefig(out, bbox_inches="tight")
    plt.close()
    print(f"  Zapisano: {out}")
 
 
def plot_size_vs_map(df: pd.DataFrame):
    """Scatter: rozmiar modelu vs mAP — efektywność parametrów."""
    fig, ax = plt.subplots(figsize=(9, 6))
 
    for _, row in df.iterrows():
        color = GEN_COLORS.get(row["Model"], "#888")
        ax.scatter(row["Rozmiar_MB"], row["mAP@0.5"],
                   color=color, s=row["Params_M"]*12, zorder=5,
                   edgecolors="white", linewidth=1.5, alpha=0.85)
        ax.annotate(row["Model"],
                    (row["Rozmiar_MB"], row["mAP@0.5"]),
                    textcoords="offset points", xytext=(6, 4),
                    fontsize=9, color=color, fontweight="bold")
 
    ax.set_xlabel("Rozmiar modelu [MB]", fontsize=12)
    ax.set_ylabel("mAP@0.5", fontsize=12)
    ax.set_title("Efektywność: rozmiar vs dokładność\n(rozmiar punktu = liczba parametrów)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = OUTPUT_DIR / "wykres_rozmiar_vs_mAP.png"
    plt.savefig(out, bbox_inches="tight")
    plt.close()
    print(f"  Zapisano: {out}")
 
 
def plot_loss_curves():
    """Training loss curves dla wszystkich modeli."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax_box, ax_cls = axes
 
    any_plotted = False
    for display_name, folder in MODELS.items():
        df = get_results_csv(folder)
        if df is None:
            continue
 
        color = GEN_COLORS.get(display_name, "#888")
 
        box_col = next((c for c in df.columns if "box" in c.lower() and "loss" in c.lower()), None)
        cls_col = next((c for c in df.columns if "cls" in c.lower() and "loss" in c.lower()), None)
 
        if box_col:
            ax_box.plot(df.index, df[box_col], label=display_name, color=color, linewidth=1.8)
        if cls_col:
            ax_cls.plot(df.index, df[cls_col], label=display_name, color=color, linewidth=1.8)
        any_plotted = True
 
    if not any_plotted:
        print("  Brak results.csv — pomijam wykresy loss.")
        return
 
    ax_box.set_title("Box Loss (trening)", fontsize=12, fontweight="bold")
    ax_box.set_xlabel("Epoka")
    ax_box.set_ylabel("Wartość straty")
    ax_box.legend(fontsize=9)
 
    ax_cls.set_title("Classification Loss (trening)", fontsize=12, fontweight="bold")
    ax_cls.set_xlabel("Epoka")
    ax_cls.set_ylabel("Wartość straty")
    ax_cls.legend(fontsize=9)
 
    plt.suptitle("Krzywe uczenia — wszystkie modele", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    out = OUTPUT_DIR / "wykres_loss_curves.png"
    plt.savefig(out, bbox_inches="tight")
    plt.close()
    print(f"  Zapisano: {out}")
 
 
def plot_radar(df: pd.DataFrame):
    """Radar chart — profil każdego modelu (normalizowane metryki)."""
    categories = ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1"]
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
 
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
 
    for _, row in df.iterrows():
        values = [row[c] for c in categories]
        values += values[:1]
        color = GEN_COLORS.get(row["Model"], "#888")
        ax.plot(angles, values, "o-", linewidth=2, color=color, label=row["Model"])
        ax.fill(angles, values, alpha=0.07, color=color)
 
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_title("Profil metryk — porównanie modeli", fontsize=13, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
 
    plt.tight_layout()
    out = OUTPUT_DIR / "wykres_radar.png"
    plt.savefig(out, bbox_inches="tight")
    plt.close()
    print(f"  Zapisano: {out}")
 
 
def save_table(df: pd.DataFrame):
    """Zapisz tabelę zbiorczą jako CSV i wersję czytelną."""
    df_sorted = df.sort_values("mAP@0.5", ascending=False).reset_index(drop=True)
    df_sorted.index += 1
 
    # CSV
    csv_path = OUTPUT_DIR / "tabela_zbiorcza.csv"
    df_sorted.to_csv(csv_path, index=True, float_format="%.4f")
    print(f"  Zapisano: {csv_path}")
 
    # Czytelne podsumowanie TXT
    txt_path = OUTPUT_DIR / "tabela_zbiorcza.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("=" * 90 + "\n")
        f.write("  YOLO MODEL COMPARISON — PODSUMOWANIE\n")
        f.write("=" * 90 + "\n\n")
        f.write(f"{'Rank':<5} {'Model':<12} {'mAP@0.5':<10} {'mAP@0.95':<10} "
                f"{'Precision':<11} {'Recall':<9} {'F1':<8} "
                f"{'FPS_CPU':<9} {'FPS_GPU':<9} {'ms_CPU':<9} {'MB':<8}\n")
        f.write("-" * 90 + "\n")
        for i, row in df_sorted.iterrows():
            gpu = f"{row['FPS_GPU']:.1f}" if pd.notna(row["FPS_GPU"]) else "N/A"
            f.write(
                f"{i:<5} {row['Model']:<12} {row['mAP@0.5']:<10.4f} "
                f"{row['mAP@0.5:0.95']:<10.4f} {row['Precision']:<11.4f} "
                f"{row['Recall']:<9.4f} {row['F1']:<8.4f} "
                f"{row['FPS_CPU']:<9.1f} {gpu:<9} "
                f"{row['Czas_ms_CPU']:<9.1f} {row['Rozmiar_MB']:<8.1f}\n"
            )
        f.write("\n")
        best = df_sorted.iloc[0]
        f.write(f"Najlepszy mAP@0.5:     {best['Model']} ({best['mAP@0.5']:.4f})\n")
        fastest = df_sorted.loc[df_sorted["FPS_CPU"].idxmax()]
        f.write(f"Najszybszy CPU:        {fastest['Model']} ({fastest['FPS_CPU']:.1f} FPS)\n")
        if df_sorted["FPS_GPU"].notna().any():
            fastest_gpu = df_sorted.loc[df_sorted["FPS_GPU"].idxmax()]
            f.write(f"Najszybszy GPU:        {fastest_gpu['Model']} ({fastest_gpu['FPS_GPU']:.1f} FPS)\n")
        smallest = df_sorted.loc[df_sorted["Rozmiar_MB"].idxmin()]
        f.write(f"Najmniejszy model:     {smallest['Model']} ({smallest['Rozmiar_MB']:.1f} MB)\n")
 
    print(f"  Zapisano: {txt_path}")
    print("\n" + open(txt_path, encoding="utf-8").read())

In [25]:
 
if __name__ == "__main__":
    print("\n" + "="*60)
    print("  YOLO Model Comparison — Start")
    print("="*60)
 
    # Sprawdź czy plik testowy istnieje
    if not Path(TEST_IMAGE).exists():
        print(f"\nUWAGA: Brak pliku testowego '{TEST_IMAGE}'.")
        print("Zmień zmienną TEST_IMAGE na ścieżkę do dowolnego zdjęcia z Twojego datasetu.\n")
 
    print("\n[1/4] Zbieranie metryk ze wszystkich modeli...")
    df = collect_metrics()
 
    if df.empty:
        print("Brak danych — sprawdź ścieżki w konfiguracji na górze skryptu.")
        exit(1)
 
    print("\n[2/4] Generowanie tabeli zbiorczej...")
    save_table(df)
 
    print("\n[3/4] Generowanie wykresów...")
    plot_map_vs_fps(df)
    plot_map_per_class(df)
    plot_fps_cpu_gpu(df)
    plot_size_vs_map(df)
    plot_radar(df)
    plot_loss_curves()
 
    print("\n[4/4] Gotowe!")
    print(f"\nWszystkie pliki zapisane w folderze: {OUTPUT_DIR.resolve()}")
    print("\nWygenerowane pliki:")
    for f in sorted(OUTPUT_DIR.iterdir()):
        print(f"  {f.name}")
 


  YOLO Model Comparison — Start

[1/4] Zbieranie metryk ze wszystkich modeli...

  Przetwarzam: YOLOv8n
  Ewaluacja na zbiorze testowym...
Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2353.8938.5 MB/s, size: 372.9 KB)
val: Scanning C:\Users\Dawid\Desktop\praca inżynierska\real-time-alaysis-research\datasets\Person&Head_DataSet\labels\val.cache... 2842 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2842/2842  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 15.9it/s 11.2s0.1s
                   all       2842      10236      0.674       0.58      0.625      0.408
Speed: 0.5ms preprocess, 0.7ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\Dawid\Desktop\praca inynierska\real-time-alaysis-research\scrip